# Prompt Engineering for Images

**Module:** 17 — Image Generation

Prompt anatomy, negatives, brand systems, and debugging when pixels refuse to listen.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Structure prompts with subject, attributes, composition, style, constraints
- Use negative prompts and weighting effectively
- Build a reusable brand prompt system
- Debug systematic failure modes


## Prompt Anatomy

### Definition
An image prompt is a structured spec: **subject + attributes + setting + composition/camera + style + constraints**.

### Why it matters
Natural language is not a camera. Structure reduces entropy and enables A/B tests.

### How it works
Write non-negotiables first; add camera/composition; finish with style; keep a negative library.

### Intuition
A film shot list — not a poem (unless the poem is the brief).

### Pitfalls
- Adjective spam without content
- Contradictory styles
- Quality words to fix composition

### When to use
Every production txt2img path.


### Anatomy template

```
[SUBJECT + COUNT], [ATTRIBUTES], [SETTING],
[COMPOSITION / LENS / LIGHT], [STYLE], [CONSTRAINTS]
```

**Tips:** concrete nouns/verbs; critical tokens early; separate style LoRAs from content; negatives for frequent artifacts.


In [ ]:
# Demo 1: prompt builder
from dataclasses import dataclass

@dataclass
class PromptSpec:
    subject: str
    attributes: str = ""
    setting: str = ""
    composition: str = ""
    style: str = ""
    constraints: str = ""
    negative: str = "blurry, watermark, lowres, deformed hands"

    def render(self) -> tuple[str, str]:
        parts = [self.subject, self.attributes, self.setting, self.composition, self.style, self.constraints]
        return ", ".join(p for p in parts if p.strip()), self.negative

spec = PromptSpec(
    subject="matte black headphones", attributes="over-ear, subtle branding",
    setting="marble surface, softbox", composition="centered product shot, 85mm",
    style="premium ecommerce photo", constraints="no text overlays",
)
print(spec.render())


In [ ]:
# Demo 2: token budget + emphasis illustration
def clip_chars(prompt: str, limit: int = 300):
    if len(prompt) <= limit: return prompt, False
    return prompt[:limit].rsplit(" ", 1)[0], True

def emphasize(prompt: str, phrase: str, weight: float = 1.2) -> str:
    return prompt.replace(phrase, f"({phrase}:{weight})") if phrase in prompt else prompt

p, trunc = clip_chars(spec.render()[0] + ", " + ", ".join(["detail"] * 80))
print("truncated", trunc, "len", len(p))
print(emphasize("a red cube and a blue sphere", "red cube", 1.3))


## Brand Systems

### Definition
A **brand prompt system** is a versioned library of style blocks, palettes, camera rules, and negatives.

### Why it matters
Consistency across hundreds of assets beats any single hero image.

### How it works
Factor `content` ⊕ `brand_style` ⊕ `channel_adaptors`; store as JSON; review diffs like code.

### Intuition
Design-system tokens for generative imagery.

### Pitfalls
- Every designer inventing style suffixes
- No golden set → drift after upgrades

### When to use
Anytime multiple people generate on-brand images.


In [ ]:
# Demo 3: brand pack composition
BRAND = {
    "version": "2026.04",
    "style_block": "flat vector, coral #FF6F61 accents, deep teal #0E4D53, generous whitespace",
    "camera": "straight-on UI mock ambience, soft diffused light",
    "negative": "photoreal clutter, neon cyberpunk, harsh HDR, watermarks",
}

def brand_prompt(content: str, brand=BRAND) -> dict:
    return {
        "prompt": f"{content}, {brand['style_block']}, {brand['camera']}",
        "negative": brand["negative"],
        "brand_version": brand["version"],
    }

print(brand_prompt("landing hero: notebook and espresso cup"))


## Failure Debugging

### Definition
Systematic diagnosis: wrong objects/counts, style drift, text errors, anatomy fails, safety false positives.

### Why it matters
Random prompt thrashing wastes budget.

### How it works
Change one variable: shorten → swap style → img2img → inpaint → seed cluster → model.

### Intuition
Hospital triage — stabilize, then specialize.

### Pitfalls
- Changing model, CFG, and prompt simultaneously
- No failure archive

### When to use
Production support and creative ops QA.


### Debugging table

| Failure | First move | Second move |
|---------|------------|-------------|
| Wrong count | Explicit count + simpler scene | Regional / two-pass |
| Style drift | Lock brand block | Reference img2img |
| Garbled text | Shorten / post-render | Text-aware model |
| Hands | Negatives + crop | Inpaint |
| Rare brand object | LoRA / boost | Product photo img2img |


In [ ]:
# Demo 4: failure ticket
def failure_ticket(prompt, symptom, params, hypothesis):
    return {"prompt": prompt, "symptom": symptom, "params": params, "hypothesis": hypothesis,
            "next_ablation": "toggle only CFG -1 and -2"}

print(failure_ticket("two red mugs on a shelf", "only one mug",
                     {"cfg": 9, "steps": 20, "model": "sdxl"}, "count token weak"))


### Try it yourself — Prompts

1. Convert an 80-word mess into the 6-slot anatomy.
2. Create brand packs for two fictional brands.
3. Categorize a negative library by artifact type.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `negative prompt` | Condition for what to avoid |
| `token weighting` | Up/down-weight phrases (stack-specific) |
| `brand block` | Reusable style/lighting/palette fragment |
| `ablation` | Change one variable to isolate cause |


### Workshop — Parameter journal — Image Prompts

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Prompts
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Prompts

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Prompts
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Prompts

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Prompts
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Prompts

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Prompts
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Prompts

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Prompts
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Prompts

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Prompts
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Prompts

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Prompts
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Image Prompts

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Image Prompts
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Structure prompts like specs; keep negatives as a library
- Brand systems are versioned infrastructure
- Debug with single-variable ablations and archived params
